# 🧠 EXACT 2026 - Track 1: Logic-Based Educational QA
### Hướng dẫn chạy thử nghiệm Pipeline trên Google Colab

Notebook này hướng dẫn bạn thiết lập môi trường và chạy thử nghiệm hệ thống Neuro-Symbolic QA (Track 1) trên Google Colab có GPU (ví dụ: T4 GPU miễn phí).

---

## 1. Kiểm tra cấu hình GPU và kết nối
Trước tiên, hãy chắc chắn rằng bạn đang sử dụng **GPU Runtime**:
- Đi tới menu: **Runtime** -> **Change runtime type**
- Chọn **T4 GPU** (hoặc GPU bất kỳ có sẵn) làm Hardware accelerator.

In [ ]:
!nvidia-smi

## 2. Clone Repository và chuyển sang nhánh `test/track1`

In [ ]:
# Clone repo từ Github
!git clone https://github.com/AIVIETNAM-AIO-Triet-Descartes/EXACT2026-NeuroSymbolic-QA.git

# Chuyển con trỏ dòng lệnh vào thư mục dự án
%cd EXACT2026-NeuroSymbolic-QA

# Chuyển sang nhánh test/track1
!git checkout test/track1

## 3. Cài đặt các thư viện phụ thuộc (Dependencies)

Chúng ta sẽ cài đặt các thư viện trong file `requirements.txt` của dự án. Đặc biệt, để sử dụng GPU tăng tốc cho mô hình Qwen GGUF, chúng ta cần biên dịch `llama-cpp-python` với tùy chọn CUDA (`GGML_CUDA=on`).

In [ ]:
# Cài đặt các thư viện chuẩn
!pip install -r requirements.txt

# Cài đặt/Biên dịch lại llama-cpp-python hỗ trợ CUDA GPU để đẩy tốc độ suy luận lên tối đa
!CMAKE_ARGS="-DGGML_CUDA=on" pip install --upgrade --force-reinstall llama-cpp-python --no-cache-dir

## 4. Tải Mô Hình Qwen 2.5 7B Instruct GGUF
Mô hình được chia nhỏ làm 2 part. Chúng ta cần tải cả 2 part về cùng một thư mục gốc. Khi chạy, thư viện `llama-cpp` sẽ tự động ghép các file này lại khi ta trỏ vào file `-00001-of-00002.gguf`.

In [ ]:
# Tải file mô hình part 1 và part 2
!wget -O qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
!wget -O qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf

## 5. Chạy thử nghiệm Pipeline (Track 1)

Để chạy pipeline, chúng ta sử dụng script `scripts/run_track1.py`.

### 5.1 Chạy thử nhanh với một lượng mẫu nhỏ (ví dụ: 5 mẫu đầu tiên)
Việc này giúp kiểm tra xem toàn bộ các thành phần (Z3 Solver, FOL Normalizer, LLM Reasoner) đã hoạt động ăn khớp với nhau hay chưa.

In [ ]:
!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_test.json \
    --model qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --max-samples 5 \
    --gpu-layers -1 \
    --evaluate

### 5.2 Chạy đánh giá trên một dải dữ liệu cụ thể (ví dụ: Mẫu 50 đến 100)
Bạn có thể thay đổi `--start-sample` và `--end-sample` để đánh giá trên các phần dữ liệu mong muốn.

In [ ]:
!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_50_100.json \
    --model qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --start-sample 50 \
    --end-sample 100 \
    --gpu-layers -1 \
    --evaluate

### 5.3 Chạy toàn bộ Dataset (411 mẫu / ~808 câu hỏi)
Thời gian chạy toàn bộ dataset trên T4 GPU sẽ mất khoảng 40 - 60 phút tùy thuộc vào hiệu suất mạng và thời gian suy luận của GPU.

In [ ]:
!PYTHONPATH=. python3 scripts/run_track1.py \
    --input Logic_Based_Educational_Queries.json \
    --output output/predictions_full.json \
    --model qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    --gpu-layers -1 \
    --evaluate

## 6. Xem Kết Quả Đầu Ra
Sau khi chạy xong, các kết quả dự đoán và đánh giá chi tiết (bao gồm cả độ chính xác Accuracy, số câu trả lời đúng/sai của mô hình) sẽ được lưu tại thư mục `output/`.

In [ ]:
# Hiển thị 5 dòng đầu của file dự đoán để kiểm tra cấu trúc
!head -n 30 output/predictions_test.json